# SAM 3 — Lab 2: Video Tracking & Advanced Analysis

## Week 10: Promptable Concept Segmentation (Part 2 of 2)

This lab continues from **Lab 1** and covers SAM 3's **video tracking** capabilities and **advanced analysis** techniques.

### What You'll Learn in This Lab
1. **Video object tracking** — bounding box, text prompt, and semantic tracking
2. **PCS vs PVS mode comparison** — understanding SAM 3's dual paradigms
3. **Interactive exemplar refinement** — improving segmentation iteratively
4. **Mask IoU & overlap analysis** — spatial relationship analysis
5. **Saving results & export** — exporting masks, bboxes, and annotated images
6. **Performance benchmarking** — measuring inference time across modes
7. **SAM 3 vs SAM 2 vs SAM** — complete feature comparison

### Architecture (Recap)
- **Detector**: DETR-based with text encoder, exemplar encoder, fusion encoder, presence head, and mask decoder
- **Tracker**: Memory-based video segmentation (inherited from SAM 2)
- **Presence Token**: Learned global token predicting concept presence

### Video Performance Highlights
| Benchmark | SAM 3 | Previous SOTA | Improvement |
|-----------|-------|---------------|-------------|
| MOSEv2 (video) | 60.1 J&F | SAM 2.1 | +25.5% |
| DAVIS 2017 | 92.0 J&F | - | - |

---
> **Prerequisite**: Complete **Lab 1** first for image segmentation fundamentals.


## 1. Environment Setup & Model Loading

Re-run the setup from Lab 1 to ensure all dependencies and the SAM 3 model are loaded.


In [ ]:
# Install/upgrade ultralytics (SAM 3 support requires latest version)
!uv pip install -U ultralytics

In [ ]:
# Core imports
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image as IPImage

# Ultralytics imports
from ultralytics import SAM
from ultralytics.models.sam import SAM3SemanticPredictor
from ultralytics.utils.plotting import Annotator, colors

# Display settings
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 100

print("All imports successful!")
print(f"OpenCV version: {cv2.__version__}")

In [ ]:
# Define paths
IMAGE_PATH = "football_teamplay.jpeg"

# Verify the image exists
assert Path(IMAGE_PATH).exists(), f"Image not found at {IMAGE_PATH}"

img = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
print(f"Image shape: {img.shape}")
print(f"Image size: {img.shape[1]}x{img.shape[0]} pixels")

In [ ]:
from huggingface_hub import login
#login(token="hf_xxxxxxxxxxxxxxxxx")  # paste your token


In [ ]:
# ── Download SAM 3 weights from Hugging Face ──────────────────────────────────
from huggingface_hub import hf_hub_download

MODEL_FILE = hf_hub_download(
    repo_id="facebook/sam3",
    filename="sam3.pt",
    cache_dir="/root/.cache/huggingface",
)
print(f"✓ {MODEL_FILE}")

In [ ]:
# Load SAM 3 model (downloads automatically if not present)
model = SAM("sam3.pt")
print(f"Model loaded: {model.info()}")

---
## 11. Video Object Tracking with Bounding Boxes

SAM 3 inherits video tracking from SAM 2 and extends it with text-based tracking. Provide initial bounding boxes on the first frame to track objects through a video.

> **Note**: This section requires a video file. A sample video download is provided, or you can use your own video.

In [ ]:
import IPython
import sys

def clean_notebook():
    IPython.display.clear_output(wait=True)
    print("Notebook cleaned.")

In [ ]:
import cv2
from IPython.display import display, Image, clear_output
import numpy as np

# Open the video file
cap = cv2.VideoCapture('.././Videos/video1.mp4')

if not cap.isOpened():
    print("Cannot open video file")
else:
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            print("Stream stopped.")
            break
        
        frame = cv2.resize(frame, (frame.shape[1] // 2, frame.shape[0] // 2))

        # Convert the frame to JPEG format for display in Jupyter
        _, buffer = cv2.imencode('.jpg', frame)
        img_bytes = buffer.tobytes()

        # Display the frame in Jupyter Notebook
        display(Image(data=img_bytes))
        clear_output(wait=True)  # Clear previous frame for smoother playback

    cap.release()
    print("Video stream ended.")


In [ ]:
# --- Experiment 11.1: Video tracking with bounding boxes ---

from ultralytics.models.sam import SAM3VideoPredictor
from huggingface_hub import hf_hub_download

# Resolve sam3.pt from HuggingFace cache (works whether or not the file
# was copied to the local directory by the download cell above)
_local_pt = Path("sam3.pt")
MODEL_PT = str(_local_pt) if _local_pt.exists() else hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt")
print(f"Using model weights: {MODEL_PT}")

# Use video1.mp4 — video5.mp4 is corrupt (moov atom missing)
VIDEO_PATH = "../../Videos/video1.mp4"

# Create a short sample video from the football image for demonstration
# (In practice, you would use a real video file)
if not Path(VIDEO_PATH).exists():
    print("Creating a synthetic video from the football image for demonstration...")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    h, w = img.shape[:2]
    VIDEO_PATH = "sample_video.mp4"
    writer = cv2.VideoWriter(VIDEO_PATH, fourcc, 10, (w, h))
    
    # Create 30 frames with slight transformations to simulate motion
    for i in range(30):
        # Simulate camera movement with translation
        M = np.float32([[1, 0, np.sin(i * 0.2) * 10], [0, 1, np.cos(i * 0.2) * 5]])
        frame = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        writer.write(frame)
    
    writer.release()
    print(f"Created sample video: {VIDEO_PATH} (30 frames)")
else:
    print(f"Using existing video: {VIDEO_PATH}")

# Video tracking with bounding box prompts
overrides_vid = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model=MODEL_PT,
    half=True,
)
vid_predictor = SAM3VideoPredictor(overrides=overrides_vid)

# Define initial bounding boxes for objects to track
# These should be on the first frame of the video
init_bboxes = [
    [350, 150, 500, 450],   # Person 1
    [550, 120, 700, 430],   # Person 2
]

print(f"\nTracking {len(init_bboxes)} objects through video...")

results = vid_predictor(
    source=VIDEO_PATH,
    bboxes=init_bboxes,
    stream=True
)

# Collect results and visualize select frames
tracked_frames = []
for frame_idx, r in enumerate(results):
    result_img = r.plot()
    tracked_frames.append(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))

# Display selected frames
n_display = min(6, len(tracked_frames))
indices = np.linspace(0, len(tracked_frames) - 1, n_display, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, idx in enumerate(indices):
    axes[i].imshow(tracked_frames[idx])
    axes[i].set_title(f"Frame {idx}", fontsize=12)
    axes[i].axis("off")

plt.suptitle("Experiment 11.1: Video Object Tracking (Bounding Box Init)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Tracked {len(tracked_frames)} frames total")


---
## 12. Video Tracking with Text Prompts (SAM 3 Exclusive)

This is a SAM 3-exclusive feature: track objects across video frames using **only text prompts**. No need to manually select bounding boxes on the first frame.

In [ ]:
# --- Experiment 12: Video tracking with text prompts ---

from ultralytics.models.sam import SAM3VideoSemanticPredictor

overrides_vid_text = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    imgsz=640,
    model="sam3.pt",
    half=True,
    save=False,
)
vid_sem_predictor = SAM3VideoSemanticPredictor(overrides=overrides_vid_text)

# Track using text prompts - no manual bbox selection needed!
print("Tracking objects using text prompts: ['person', 'ball']")

results = vid_sem_predictor(
    source=VIDEO_PATH,
    text=["person", "ball"],
    stream=True
)

# Collect and visualize
text_tracked_frames = []
for r in results:
    result_img = r.plot()
    text_tracked_frames.append(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))

# Display selected frames
n_display = min(6, len(text_tracked_frames))
indices = np.linspace(0, len(text_tracked_frames) - 1, n_display, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, idx in enumerate(indices):
    axes[i].imshow(text_tracked_frames[idx])
    axes[i].set_title(f"Frame {idx}", fontsize=12)
    axes[i].axis("off")

plt.suptitle('Experiment 12: Text-Based Video Tracking ["person", "ball"]', fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Text-tracked {len(text_tracked_frames)} frames")

---
## 13. Video Semantic Tracking with Bounding Boxes + Labels

Combine bounding boxes with label IDs for video tracking. Assign label IDs to group objects by category.

In [ ]:
# --- Experiment 13: Video tracking with bboxes and labels ---

vid_sem_predictor2 = SAM3VideoSemanticPredictor(overrides=overrides_vid_text)

# Track with bounding boxes + labels
# labels group objects by category (same label = same class)
init_bboxes_labeled = [
    [350, 150, 500, 450],   # Person 1
    [550, 120, 700, 430],   # Person 2
]
init_labels = [1, 1]  # Both are "person" class (label 1)

print(f"Tracking {len(init_bboxes_labeled)} objects with labels: {init_labels}")

results = vid_sem_predictor2(
    source=VIDEO_PATH,
    bboxes=init_bboxes_labeled,
    labels=init_labels,
    stream=True
)

labeled_frames = []
for r in results:
    result_img = r.plot()
    labeled_frames.append(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))

# Display first and last frames
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(labeled_frames[0])
axes[0].set_title("First Frame (Initialization)", fontsize=14)
axes[0].axis("off")

axes[1].imshow(labeled_frames[-1])
axes[1].set_title("Last Frame (Tracked)", fontsize=14)
axes[1].axis("off")

plt.suptitle("Experiment 13: Bbox + Label Video Tracking", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 14. Comparing PCS (SAM 3) vs PVS (SAM 2-compatible) Modes

SAM 3 supports two segmentation paradigms:
- **PCS (Promptable Concept Segmentation)**: Text/exemplar prompts -> finds ALL instances (SAM 3 new)
- **PVS (Promptable Visual Segmentation)**: Point/box prompts -> segments SINGLE object (SAM 2 compatible)

In [ ]:
# --- Experiment 14: PCS vs PVS comparison ---

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# --- PCS Mode (SAM 3 exclusive): finds ALL people ---
pcs_overrides = dict(conf=0.25, task="segment", mode="predict", model="sam3.pt", half=True, save=False)
pcs_pred = SAM3SemanticPredictor(overrides=pcs_overrides)
pcs_pred.set_image(IMAGE_PATH)
pcs_results = pcs_pred(text=["person"])

pcs_img = pcs_results[0].plot()
pcs_img_rgb = cv2.cvtColor(pcs_img, cv2.COLOR_BGR2RGB)
n_pcs = len(pcs_results[0].masks) if pcs_results[0].masks is not None else 0

axes[0].imshow(pcs_img_rgb)
axes[0].set_title(f'PCS: text="person"\n({n_pcs} instances found)', fontsize=13)
axes[0].axis("off")

# --- PVS Mode (SAM 2 compatible): single object at point ---
pvs_model = SAM("sam3.pt")
pvs_results = pvs_model.predict(source=IMAGE_PATH, points=[400, 300], labels=[1])

pvs_img = pvs_results[0].plot()
pvs_img_rgb = cv2.cvtColor(pvs_img, cv2.COLOR_BGR2RGB)

axes[1].imshow(pvs_img_rgb)
axes[1].set_title("PVS: point=[400,300]\n(1 object segmented)", fontsize=13)
axes[1].axis("off")

# --- PVS Mode: box prompt ---
pvs_box_results = pvs_model.predict(source=IMAGE_PATH, bboxes=[350, 130, 520, 460])

pvs_box_img = pvs_box_results[0].plot()
pvs_box_img_rgb = cv2.cvtColor(pvs_box_img, cv2.COLOR_BGR2RGB)

axes[2].imshow(pvs_box_img_rgb)
axes[2].set_title("PVS: box=[350,130,520,460]\n(1 object segmented)", fontsize=13)
axes[2].axis("off")

plt.suptitle("Experiment 14: PCS (All Instances) vs PVS (Single Object)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

print("Key Difference:")
print(f"  PCS (text='person'): Found ALL {n_pcs} people in the image")
print(f"  PVS (point/box):     Segments only the SINGLE object at the prompt location")

---
## 15. Interactive Exemplar Refinement

SAM 3's performance improves significantly with more exemplars:
- Text only: **46.4** CGF1
- +1 exemplar: **57.6** CGF1  
- +3 exemplars: **65.0** CGF1 (+18.6 gain)

Let's demonstrate this iterative improvement.

In [ ]:
# --- Experiment 15: Interactive refinement with increasing exemplars ---

exemplar_sets = [
    {"name": "Text only", "text": ["person"], "bboxes": None},
    {"name": "+1 exemplar", "text": None, "bboxes": [[350, 150, 500, 450]]},
    {"name": "+2 exemplars", "text": None, "bboxes": [[350, 150, 500, 450], [550, 120, 700, 430]]},
]

ref_overrides = dict(conf=0.25, task="segment", mode="predict", model="sam3.pt", half=True, save=False)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for idx, config in enumerate(exemplar_sets):
    pred_ref = SAM3SemanticPredictor(overrides=ref_overrides)
    pred_ref.set_image(IMAGE_PATH)
    
    if config["text"]:
        results = pred_ref(text=config["text"])
    else:
        results = pred_ref(bboxes=config["bboxes"])
    
    n_found = len(results[0].masks) if results[0].masks is not None else 0
    
    result_img = results[0].plot()
    result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(result_img_rgb)
    axes[idx].set_title(f'{config["name"]}\n({n_found} found)', fontsize=13)
    axes[idx].axis("off")

plt.suptitle("Experiment 15: Iterative Refinement with Exemplars", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

print("Interactive refinement improves detection quality:")
print("  Text only:      46.4 CGF1 (baseline)")
print("  +1 exemplar:    57.6 CGF1 (+11.2)")
print("  +3 exemplars:   65.0 CGF1 (+18.6 total gain)")

---
## 16. Mask IoU and Overlap Analysis

Analyze the spatial relationships between detected masks - useful for understanding scene layout and occlusions.

In [ ]:
# --- Experiment 16: Mask IoU and overlap analysis ---

iou_overrides = dict(conf=0.25, task="segment", mode="predict", model="sam3.pt", half=True, save=False)
pred_iou = SAM3SemanticPredictor(overrides=iou_overrides)
pred_iou.set_image(IMAGE_PATH)

results = pred_iou(text=["person"])

if results[0].masks is not None and len(results[0].masks) > 1:
    masks = results[0].masks.data.cpu().numpy()
    n = len(masks)
    
    # Compute pairwise IoU
    iou_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            intersection = np.logical_and(masks[i], masks[j]).sum()
            union = np.logical_or(masks[i], masks[j]).sum()
            iou_matrix[i, j] = intersection / union if union > 0 else 0
    
    # Visualize IoU matrix
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Heatmap
    im = axes[0].imshow(iou_matrix, cmap="YlOrRd", vmin=0, vmax=1)
    axes[0].set_title("Pairwise Mask IoU", fontsize=14)
    axes[0].set_xlabel("Person ID")
    axes[0].set_ylabel("Person ID")
    axes[0].set_xticks(range(n))
    axes[0].set_yticks(range(n))
    axes[0].set_xticklabels([f"P{i+1}" for i in range(n)])
    axes[0].set_yticklabels([f"P{i+1}" for i in range(n)])
    
    # Add text annotations
    for i in range(n):
        for j in range(n):
            axes[0].text(j, i, f"{iou_matrix[i,j]:.2f}", ha="center", va="center",
                        color="white" if iou_matrix[i,j] > 0.5 else "black", fontsize=9)
    
    plt.colorbar(im, ax=axes[0], label="IoU")
    
    # Mask overlap visualization
    overlap_img = np.zeros((*masks[0].shape, 3), dtype=np.uint8)
    colormap = plt.cm.Set2(np.linspace(0, 1, n))[:, :3]
    
    for i in range(n):
        mask_bool = masks[i].astype(bool)
        color = (colormap[i] * 255).astype(np.uint8)
        overlap_img[mask_bool] = (overlap_img[mask_bool] * 0.5 + color * 0.5).astype(np.uint8)
    
    axes[1].imshow(overlap_img)
    axes[1].set_title("Mask Overlap Visualization", fontsize=14)
    axes[1].axis("off")
    
    plt.suptitle("Experiment 16: Mask IoU & Overlap Analysis", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
    
    # Print overlap statistics
    print(f"\nOverlap Statistics for {n} detected persons:")
    for i in range(n):
        for j in range(i+1, n):
            if iou_matrix[i, j] > 0:
                print(f"  Person {i+1} <-> Person {j+1}: IoU = {iou_matrix[i,j]:.4f}")
else:
    print("Need at least 2 masks for overlap analysis")

---
## 17. Saving Results & Export

Export segmentation results for downstream applications: save masks, bounding boxes, and annotated images.

In [ ]:
# --- Experiment 17: Save results to disk ---

output_dir = Path("sam3_results")
output_dir.mkdir(exist_ok=True)

save_overrides = dict(conf=0.25, task="segment", mode="predict", model="sam3.pt", half=True, save=False)
pred_save = SAM3SemanticPredictor(overrides=save_overrides)
pred_save.set_image(IMAGE_PATH)

results = pred_save(text=["person"])

if results[0].masks is not None:
    masks = results[0].masks.data.cpu().numpy()
    boxes = results[0].boxes.data.cpu().numpy()
    
    # 1. Save annotated image
    annotated = results[0].plot()
    cv2.imwrite(str(output_dir / "annotated_persons.jpg"), annotated)
    print(f"Saved: {output_dir / 'annotated_persons.jpg'}")
    
    # 2. Save individual binary masks
    for i, mask in enumerate(masks):
        mask_uint8 = (mask * 255).astype(np.uint8)
        cv2.imwrite(str(output_dir / f"mask_person_{i+1}.png"), mask_uint8)
    print(f"Saved: {len(masks)} individual mask PNGs")
    
    # 3. Save combined mask
    combined = np.any(masks, axis=0).astype(np.uint8) * 255
    cv2.imwrite(str(output_dir / "mask_all_persons.png"), combined)
    print(f"Saved: {output_dir / 'mask_all_persons.png'}")
    
    # 4. Save bounding boxes to text file
    with open(output_dir / "detections.txt", "w") as f:
        f.write("# person_id x1 y1 x2 y2 confidence\n")
        for i, box in enumerate(boxes):
            f.write(f"{i+1} {box[0]:.1f} {box[1]:.1f} {box[2]:.1f} {box[3]:.1f} {box[4]:.4f}\n")
    print(f"Saved: {output_dir / 'detections.txt'}")
    
    # 5. Save extracted objects (transparent PNG)
    for i, mask in enumerate(masks):
        mask_bool = mask.astype(bool)
        # Create RGBA image
        rgba = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)
        rgba[~mask_bool, 3] = 0  # Set alpha to 0 for non-mask pixels
        
        # Crop to mask bounds
        rows_m = np.any(mask_bool, axis=1)
        cols_m = np.any(mask_bool, axis=0)
        rmin, rmax = np.where(rows_m)[0][[0, -1]]
        cmin, cmax = np.where(cols_m)[0][[0, -1]]
        cropped_rgba = rgba[rmin:rmax+1, cmin:cmax+1]
        
        cv2.imwrite(str(output_dir / f"extracted_person_{i+1}.png"), cropped_rgba)
    print(f"Saved: {len(masks)} extracted person PNGs (transparent background)")

print(f"\nAll results saved to: {output_dir.absolute()}")
print(f"Files: {sorted([f.name for f in output_dir.iterdir()])}")

---
## 18. Performance Benchmarking

Measure inference time for different SAM 3 operations.

In [ ]:
# --- Experiment 18: Performance benchmarking ---
import time

bench_overrides = dict(conf=0.25, task="segment", mode="predict", model="sam3.pt", half=True, save=False)

benchmarks = {}

# 1. Text prompt (PCS)
pred_bench = SAM3SemanticPredictor(overrides=bench_overrides)
times = []
for _ in range(3):
    start = time.time()
    pred_bench.set_image(IMAGE_PATH)
    results = pred_bench(text=["person"])
    times.append(time.time() - start)
benchmarks["Text: 'person'"] = np.mean(times)

# 2. Multiple text prompts
times = []
for _ in range(3):
    start = time.time()
    pred_bench.set_image(IMAGE_PATH)
    results = pred_bench(text=["person", "ball", "tree"])
    times.append(time.time() - start)
benchmarks["Text: multi-concept"] = np.mean(times)

# 3. Exemplar bbox
times = []
for _ in range(3):
    start = time.time()
    pred_bench.set_image(IMAGE_PATH)
    results = pred_bench(bboxes=[[350, 150, 500, 450]])
    times.append(time.time() - start)
benchmarks["Exemplar: 1 bbox"] = np.mean(times)

# 4. Point prompt (PVS)
model_bench = SAM("sam3.pt")
times = []
for _ in range(3):
    start = time.time()
    results = model_bench.predict(source=IMAGE_PATH, points=[400, 300], labels=[1])
    times.append(time.time() - start)
benchmarks["Point: single"] = np.mean(times)

# 5. Box prompt (PVS)
times = []
for _ in range(3):
    start = time.time()
    results = model_bench.predict(source=IMAGE_PATH, bboxes=[350, 130, 520, 460])
    times.append(time.time() - start)
benchmarks["Box: single"] = np.mean(times)

# Visualize benchmarks
plt.figure(figsize=(12, 5))
names = list(benchmarks.keys())
times_ms = [v * 1000 for v in benchmarks.values()]

bars = plt.barh(names, times_ms, color=["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"])
plt.xlabel("Time (ms)", fontsize=13)
plt.title("Experiment 18: Inference Time Comparison", fontsize=16, fontweight="bold")

for bar, t in zip(bars, times_ms):
    plt.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             f"{t:.0f} ms", va="center", fontsize=11)

plt.tight_layout()
plt.show()

print("\nBenchmark Results (averaged over 3 runs):")
for name, t in benchmarks.items():
    print(f"  {name}: {t*1000:.0f} ms")

---
## 19. SAM 3 with Auto-Save Mode

Use `save=True` in overrides to automatically save annotated results to `runs/segment/predict/`.

In [ ]:
# --- Experiment 19: Auto-save mode ---

auto_save_overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="sam3.pt",
    half=True,
    save=True,  # Automatically saves annotated results
)

pred_auto = SAM3SemanticPredictor(overrides=auto_save_overrides)
pred_auto.set_image(IMAGE_PATH)

# Run segmentation - results will be auto-saved
results = pred_auto(text=["person", "ball"])

print("Results automatically saved by ultralytics to runs/segment/predict/")
print("This is useful for batch processing multiple images.")

# Check save location
from pathlib import Path
save_dirs = sorted(Path("runs").rglob("*.jpg")) if Path("runs").exists() else []
if save_dirs:
    print(f"\nSaved files found:")
    for f in save_dirs[:5]:
        print(f"  {f}")

---
## 20. Summary & Feature Comparison

### SAM 3 vs SAM 2 vs SAM - Complete Feature Matrix

In [ ]:
# --- Summary: Feature comparison table ---
import pandas as pd

comparison = {
    "Feature": [
        "All instances detection (PCS)",
        "Text prompts",
        "Image exemplars",
        "Point prompts",
        "Box prompts",
        "Mask prompts",
        "Single object segmentation (PVS)",
        "Video tracking",
        "Text-based video tracking",
        "Presence head",
        "Open vocabulary",
        "Object counting",
        "Interactive refinement",
    ],
    "SAM 3": ["Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes"],
    "SAM 2": ["No", "No", "No", "Yes", "Yes", "Yes", "Yes", "Yes", "No", "No", "No", "No", "No"],
    "SAM": ["No", "No", "No", "Yes", "Yes", "Yes", "Yes", "No", "No", "No", "No", "No", "No"],
}

df = pd.DataFrame(comparison)
df = df.set_index("Feature")

# Style the table
def highlight_yes(val):
    if val == "Yes":
        return "background-color: #d4edda; color: #155724; font-weight: bold"
    return "background-color: #f8d7da; color: #721c24"

styled = df.style.applymap(highlight_yes)
display(styled)

In [ ]:
# --- Summary: Benchmark comparison chart ---

benchmarks_data = {
    "Benchmark": ["LVIS\n(zero-shot)", "SA-Co/Gold\n(CGF1)", "COCO\n(Box AP)", "MOSEv2\n(J&F)", "DAVIS 2017\n(J&F)", "CountBench\n(Accuracy)"],
    "SAM 3": [47.0, 65.0, 53.5, 60.1, 92.0, 95.6],
    "Previous SOTA": [38.5, 34.3, None, None, None, None],
}

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(benchmarks_data["Benchmark"]))
width = 0.35

bars1 = ax.bar(x - width/2, benchmarks_data["SAM 3"], width, label="SAM 3", color="#4ECDC4", edgecolor="white")

# Only plot previous SOTA where available
prev_vals = [v if v is not None else 0 for v in benchmarks_data["Previous SOTA"]]
prev_colors = ["#FF6B6B" if v is not None else "none" for v in benchmarks_data["Previous SOTA"]]
bars2 = ax.bar(x + width/2, prev_vals, width, label="Previous SOTA",
               color=prev_colors, edgecolor="white")

ax.set_ylabel("Score", fontsize=13)
ax.set_title("SAM 3 Performance Benchmarks", fontsize=16, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(benchmarks_data["Benchmark"], fontsize=10)
ax.legend(fontsize=12)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f"{height:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

for bar, val in zip(bars2, benchmarks_data["Previous SOTA"]):
    if val is not None:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=10)

ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

---
## Experiment Index

| # | Experiment | Feature Tested | Section |
|---|-----------|----------------|---------|
| 3.1 | Single text prompt | PCS - text segmentation | Text-Based |
| 3.2 | Multiple text prompts | PCS - multi-concept | Text-Based |
| 3.3 | Descriptive text prompts | PCS - noun phrases | Text-Based |
| 3.4 | Batch multi-concept | PCS - batch inference | Text-Based |
| 4.1 | Coordinate reference | Image exploration | Exemplar |
| 4.2 | Single bbox exemplar | PCS - exemplar | Exemplar |
| 4.3 | Multiple bbox exemplars | PCS - refinement | Exemplar |
| 5.1 | Single point prompt | PVS - point | Point Prompt |
| 5.2 | Multiple point prompts | PVS - multi-point | Point Prompt |
| 5.3 | FG + BG points | PVS - fg/bg labels | Point Prompt |
| 6.1 | Box prompt | PVS - box | Box Prompt |
| 7 | Confidence threshold sweep | Threshold tuning | Analysis |
| 8 | Feature-based inference | Efficient multi-query | Efficiency |
| 8b | Custom annotator | Visualization | Efficiency |
| 9.1 | Mask analysis | Individual masks | Counting |
| 9.2 | Multi-concept counting | Object counting | Counting |
| 10.1 | Object extraction | Background removal | Extraction |
| 10.2 | Semantic colormap | Scene understanding | Extraction |
| 11.1 | Video tracking (bbox) | Video + bbox init | Video |
| 12 | Video tracking (text) | Video + text init | Video |
| 13 | Video tracking (bbox+labels) | Video + labeled bbox | Video |
| 14 | PCS vs PVS comparison | Mode comparison | Comparison |
| 15 | Iterative refinement | Exemplar improvement | Refinement |
| 16 | IoU & overlap analysis | Mask relationships | Analysis |
| 17 | Export results | Save masks/boxes/images | Export |
| 18 | Performance benchmarks | Timing comparison | Benchmarks |
| 19 | Auto-save mode | Built-in saving | Utility |
| 20 | Feature comparison | SAM 3 vs SAM 2 vs SAM | Summary |

---
### References
- [Ultralytics SAM 3 Documentation](https://docs.ultralytics.com/models/sam-3/)
- Meta AI: SAM 3 - Segment Anything Model 3 for Promptable Concept Segmentation
- Model: `sam3.pt` (473.6M parameters, 3.45 GB)